## Deconstructing QA data

In [ ]:
import stanza as snlp
import pandas as pd
import json

In [ ]:
df = pd.read_csv("data/nextgqa/train.csv") # csv file of the train set
df = df[["question", "answer"]]
df

In [ ]:
nlp = snlp.Pipeline()
data = df.to_dict(orient='records')

In [ ]:
# Use StanfordNLP for text analysis and map construction
def process_text(text):
    doc = nlp(text)
    processed_data = []

    for sentence in doc.sentences:
        for word in sentence.words:
            # token = sentence.tokens[word.index - 1]
            processed_data.append({
                'text': word.text,
                'lemma': word.lemma,
                'upos': word.upos,
                'xpos': word.xpos,
                'head': word.head,  # head index (1-based)
                'deprel': word.deprel
            })

    return processed_data

def extract_components(question_data, answer_data):
    # Question type: Usually a question word
    question_type = next((word['text'] for word in question_data if word['upos'] in ['PRON', 'ADV', 'DET']), None)

    # Subject in question: 'nsubj' (noun subject) or 'nsubjpass' (passive noun subject) in dependency
    subject = next((word['text'] for word in question_data if word['deprel'] in ['nsubj', 'nsubjpass']), None)

    # All verbs in question
    verbs = list(set(word['text'] for word in question_data if word['upos'] == 'VERB'))

    # Object in question: all entities except the subject
    objs = list(set(word['text'] for word in question_data if word['deprel'] not in ['nsubj', 'nsubjpass'] and word['upos'] in ['NOUN', 'PROPN', 'PROPN', 'ADJ']))

    # Answers set
    answer = ' '.join([word['text'] for word in answer_data])

    # a_subject = next((word['text'] for word in answer_data if word['deprel'] in ['nsubj', 'nsubjpass']), None)

    # All verbs in the answer
    a_verbs = list(set(word['text'] for word in answer_data if word['upos'] == 'VERB'))

    # Object in the answer: all entities except the subject
    a_objs = list(set(word['text'] for word in answer_data if word['deprel'] not in ['nsubj', 'nsubjpass'] and word['upos'] in ['NOUN', 'PROPN']))

    components = {
        'question': ' '.join([word['text'] for word in question_data]),
        'question_type': question_type,
        'subject': subject,
        'verbs': verbs,
        'objects': objs,
        'answer': answer,
        'answer_verbs': a_verbs,
        'answer_objects': a_objs,
    }

    return components

In [ ]:
import json
import os
import gc

# Đường dẫn để lưu file tạm (Nhớ đổi thành nextgqa nếu bạn đang chạy bộ NExTGQA)
save_path = 'data/nextgqa/causal_feature/data_components.json'

# 1. Cơ chế khôi phục (Resume): Kiểm tra xem trước đó đã chạy được bao nhiêu rồi
if os.path.exists(save_path):
    with open(save_path, 'r') as f:
        data_components = json.load(f)
    start_index = len(data_components)
    print(f"Phát hiện file cũ. Bắt đầu chạy tiếp từ mốc: {start_index}/{len(data)}")
else:
    data_components = []
    start_index = 0

# 2. Chạy vòng lặp
for i in range(start_index, len(data)):
    question_data = process_text(data[i]['question'])
    answer_data = process_text(data[i]['answer'])
    components = extract_components(question_data, answer_data)
    data_components.append(components)

    # In tiến độ (In mỗi 50 bước để tránh việc in \r quá nhanh làm đơ trình duyệt)
    if (i + 1) % 50 == 0 or (i + 1) == len(data):
        print(f"\rProgress: {i+1}/{len(data)}", end='')

    # 3. Cơ chế Checkpoint: Cứ mỗi 2000 mẫu thì lưu ổ cứng và dọn dẹp RAM 1 lần
    if (i + 1) % 2000 == 0:
        with open(save_path, 'w') as f:
            json.dump(data_components, f, indent=4)

        # Ép Python dọn dẹp bộ nhớ RAM rác
        gc.collect()

# 4. Lưu lần cuối khi hoàn thành toàn bộ vòng lặp
with open(save_path, 'w') as f:
    json.dump(data_components, f, indent=4)
print("\nĐã xử lý và lưu hoàn tất!")

## encoding the components

In [ ]:
def encode_text_with_clip(text, model, processor):
    # 1. Biến text thành tensor (lúc này đang ở CPU)
    inputs = processor(text=text, return_tensors="pt", padding=True, truncation=True)

    # 2. DÒNG QUAN TRỌNG NHẤT: Bốc toàn bộ input đẩy lên cùng nhà (GPU) với model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # 3. Tính toán và sinh đặc trưng
    with torch.no_grad():
        text_features = model.get_text_features(**inputs)

    # 4. Kéo kết quả trả về CPU để dành VRAM cho các tác vụ khác
    return text_features.cpu()

def encode_components(components, model, processor):
    encoded_components = {}
    for key, value in components.items():
        if isinstance(value, list):
            # Encode list of texts
            text = ' '.join(value)
        else:
            # Encode single text
            text = value if value else ''
        encoded_components[key] = encode_text_with_clip(text, model, processor)
    return encoded_components

In [ ]:
from transformers import CLIPProcessor, CLIPModel
import torch

# Kiểm tra xem Colab đang có GPU không
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Đang sử dụng thiết bị: {device}")

# Tải phiên bản CLIP tiêu chuẩn thường được dùng cho các tác vụ VideoQA
# Đổi "base-patch32" thành "large-patch14"
model_name = "openai/clip-vit-large-patch14"
clip_processor = CLIPProcessor.from_pretrained(model_name)
clip_model = CLIPModel.from_pretrained(model_name).to(device)

In [ ]:
import torch
encoded_data_components = []
for i in range(len(data)):
    components = data_components[i]
    encoded_components = encode_components(components, clip_model, clip_processor)
    encoded_data_components.append(encoded_components)
    print(f"\rProgress: {i+1}/{len(data)}", end='')
# 将结果保存为npy文件
torch.save(encoded_data_components, 'data/nextgqa/causal_feature/encoded_data_components.npy')

## Constructing the Graph

In [ ]:
import torch_geometric
from torch_geometric.data import Data

def build_knowledge_graph(encoded_components):
    nodes = []
    edges = []

    # Define node features
    nodes.append(encoded_components['question'])
    nodes.append(encoded_components['question_type'])
    nodes.append(encoded_components['subject'])
    nodes.append(encoded_components['verbs'])
    nodes.append(encoded_components['objects'])
    nodes.append(encoded_components['answer'])
    nodes.append(encoded_components['answer_verbs'])
    nodes.append(encoded_components['answer_objects'])

    # Define edges
    edge_index = []

    # Add edges according to the specified relationships
    edge_index.extend([
        (0, 5),  # q--a
        (0, 1),  # q--q_type
        (0, 2),  # q--q_sub
        (0, 3),  # q--q_verb
        (0, 4),  # q--q_obj
        (2, 4),  # q_sub--q_obj
        (2, 3),  # q_sub--q_verb
        (4, 3),  # q_obj--q_verb
        (5, 6),  # a--a_verb
        (5, 7),  # a--a_obj
        (6, 7),  # a_verb--a_obj
        (2, 5)   # q_sub--a
    ])

    # Convert edge_index to tensor
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

    # Convert nodes to tensor
    x = torch.cat(nodes, dim=0)

    # Create PyTorch Geometric Data object
    data = Data(x=x, edge_index=edge_index)

    return data

In [ ]:
graphs = []
for i in range(len(encoded_data_components)):
    encoded_components = encoded_data_components[i]
    knowledge_graph = build_knowledge_graph(encoded_components)
    graphs.append(knowledge_graph)

## Clustering graph

In [ ]:
from sklearn.cluster import MiniBatchKMeans
import numpy as np
from torch_geometric.data import Data

def kmean(x, k=512):
    x = torch.tensor(x)
    x = x.numpy()

    # SỬA Ở ĐÂY: Đổi 768 thành 512
    x = x.reshape([-1, 768*8])

    # Apply K-means algorithm
    print("feature sample:", x.shape[0])
    kmeans = MiniBatchKMeans(n_clusters=k, random_state=43, verbose=True).fit(x)
    print("clustering done")
    # Get the cluster center point
    cluster_centers = kmeans.cluster_centers_
    print("Get centers")
    # Gets the cluster label for each point
    labels = kmeans.labels_

    # Initializes a list to store the member characteristics of each cluster
    cluster_features = [x[labels == i] for i in range(k)]
    # Calculate the feature mean of each cluster
    cluster_means = [np.mean(cluster, axis=0) for cluster in cluster_features]
    print("Get mean")
    return cluster_centers, cluster_means

def cluster_knowledge_graphs(graphs, n_clusters=512, batch_size=100):
    # Flatten all node features into a single matrix
    all_features = []
    for graph in graphs:
        all_features.append(graph.x.reshape(1, -1))

    all_features = torch.cat(all_features, dim=0).numpy()

    # Perform MiniBatchKMeans clustering
    cluster_centers, cluster_means = kmean(all_features)

    # SỬA Ở ĐÂY: Đổi 768 thành 512
    cluster_centers = torch.from_numpy(cluster_centers).reshape(-1, 8, 768)
    cluster_means = torch.tensor(cluster_means).reshape(-1, 8, 768)

    # Reorganize graphs based on clustering results
    # Define edges
    edge_index = []

    # Add edges according to the specified relationships
    edge_index.extend([
        (0, 5),  # q--a
        (0, 1),  # q--q_type
        (0, 2),  # q--q_sub
        (0, 3),  # q--q_verb
        (0, 4),  # q--q_obj
        (2, 4),  # q_sub--q_obj
        (2, 3),  # q_sub--q_verb
        (4, 3),  # q_obj--q_verb
        (5, 6),  # a--a_verb
        (5, 7),  # a--a_obj
        (6, 7),  # a_verb--a_obj
        (2, 5)   # q_sub--a
    ])

    # Convert edge_index to tensor
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

    # Create PyTorch Geometric Data object
    clustered_centers_graphs = Data(x=cluster_centers, edge_index=edge_index)
    clustered_means_graphs = Data(x=cluster_means, edge_index=edge_index)

    return clustered_centers_graphs, clustered_means_graphs

In [ ]:
clustered_centers_graphs, clustered_means_graphs = cluster_knowledge_graphs(graphs, n_clusters=512, batch_size=100)

In [ ]:
torch.save({"k_center": clustered_centers_graphs, "k_mean": clustered_means_graphs}, "data/nextgqa/causal_feature/qa_graphs.npy")

## visual feature

In [ ]:
import h5py
import torch
import numpy as np
from sklearn.cluster import KMeans, MiniBatchKMeans

In [ ]:
def kmean(x, k=512):
    # Lỗi 1: Đổi 2048 của ResNet thành 512 của CLIPL
    x = x.reshape([-1, 768])
    print("feature sample:", x.shape[0])

    # Lỗi 2: Thêm n_init='auto' để triệt tiêu cảnh báo đỏ Future Warning của sklearn
    kmeans = MiniBatchKMeans(n_clusters=k, random_state=43, verbose=True, n_init='auto').fit(x)
    print("clustering done")
    cluster_centers = kmeans.cluster_centers_
    print("Get centers")
    labels = kmeans.labels_

    cluster_features = [x[labels == i] for i in range(k)]
    # Đảm bảo xử lý an toàn nếu có cụm nào vô tình bị rỗng
    cluster_means = [np.mean(cluster, axis=0) if len(cluster) > 0 else np.zeros(512) for cluster in cluster_features]
    print("Get mean")
    return cluster_centers, cluster_means

In [ ]:
def make_sample(path="/content/data/nextqa/video_feature/CLIPL/train.h5"): # BẠN NHỚ SỬA TÊN FILE H5 Ở ĐÂY CHO ĐÚNG
    frame_feats = []
    with h5py.File(path, 'r') as fp:
        vids = fp['vid']

        # Tên cột dữ liệu của CLIP thường là 'clip' hoặc 'TempCLIP', không phải 'ResNet_I'
        # Đoạn code này sẽ tự động dò tìm key chứa feature để tránh lỗi ngớ ngẩn
        feat_key = [k for k in fp.keys() if k != 'vid'][0]
        print(f"Đang đọc dữ liệu từ cột: {feat_key}...")

        feats = fp[feat_key]

        for id, (vid, feat) in enumerate(zip(vids, feats)):
            # Bỏ qua lỗi decode nếu vid đã là kiểu string
            if isinstance(vid, bytes):
                vid = vid.decode("utf-8")
            frame_feats.append(feat)

    frame_feats = np.array(frame_feats)

    # Tiến hành gom cụm
    k_center, k_mean = kmean(frame_feats)

    k_center = torch.from_numpy(k_center)

    # Lỗi 3: Gói mảng vào np.array() trước khi chuyển sang tensor để khắc phục cảnh báo extremely slow
    k_mean = torch.tensor(np.array(k_mean))

    # SỬA ĐƯỜNG DẪN LƯU FILE: Trỏ thẳng về thư mục causal_feature của NExTGQA
    save_path = "data/nextgqa/causal_feature/visual_clusters.npy"
    torch.save({"k_center": k_center, "k_mean": k_mean}, save_path)
    print(f"\nĐã lưu thành công đặc trưng thị giác V^ tại: {save_path}")

# Chạy hàm
make_sample()

In [ ]:
# 1. Tạo thư mục CLIPL (nếu chưa có)
!mkdir -p /content/data/nextqa/video_feature/CLIPL/

# 2. Đổi tên và di chuyển file train
!mv /content/data/nextqa/video_feature/CLIPL_I_train.h5 /content/data/nextqa/video_feature/CLIPL/train.h5

# 3. Làm tương tự cho file val (Validation) và test (Để tránh code chạy xong epoch 1 đi đánh giá lại bị crash)
!mv /content/data/nextqa/video_feature/CLIPL_I_val.h5 /content/data/nextqa/video_feature/CLIPL/val.h5
!mv /content/data/nextqa/video_feature/CLIPL_I_test.h5 /content/data/nextqa/video_feature/CLIPL/test.h5